# Foundational: Amazon Polly Text-to-Speech

Converts text to audio using Amazon Polly and uploads the result to S3.

> Running inside SageMaker — IAM role credentials are used automatically via the instance metadata service. No need to set `AWS_ACCESS_KEY_ID` or `AWS_SECRET_ACCESS_KEY`.

**Required IAM permissions on your SageMaker execution role:**
- `polly:SynthesizeSpeech`
- `s3:PutObject` on your target bucket

## 1. Configuration

In [ ]:
# ── Set these before running ─────────────────────────────────────────────────
S3_BUCKET_NAME = "your-bucket-name"   # <-- replace
OUTPUT_KEY     = "polly-audio/output.mp3"
AWS_REGION     = "us-east-1"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Define the Text to Synthesize

In [ ]:
speech_text = """
Welcome to Pixel Learning Co.
This course will guide you through the fundamentals of cloud computing and AI services.
We believe that learning should be accessible to everyone, everywhere, at any time.
Let's begin your journey today.
""".strip()

## 3. Synthesize with Amazon Polly and Upload to S3

In [ ]:
import boto3

polly = boto3.client("polly", region_name=AWS_REGION)

response = polly.synthesize_speech(
    Text=speech_text,
    OutputFormat="mp3",
    VoiceId="Joanna",
    Engine="neural",
)

audio_bytes = response["AudioStream"].read()
print(f"Received {len(audio_bytes):,} bytes of audio from Polly")

In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION)
s3.put_object(
    Bucket=S3_BUCKET_NAME,
    Key=OUTPUT_KEY,
    Body=audio_bytes,
    ContentType="audio/mpeg",
)
print(f"Uploaded to s3://{S3_BUCKET_NAME}/{OUTPUT_KEY}")

## 4. Preview the Audio (optional)

In [ ]:
import IPython.display as ipd
ipd.Audio(audio_bytes, rate=22050)